In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

path = 'power_customer_energy.csv'
df = pd.read_csv(path)
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())

In [ ]:
num = df.select_dtypes('number')
num = num.drop(columns=[c for c in num.columns if c.lower() in ['unnamed: 0', 'id', 'index', 'customer_id']], errors='ignore')
num = num.dropna()

target = None
for key in ['consumption', 'kwh', 'energy', 'usage']:
    match = [c for c in num.columns if key in c.lower()]
    if len(match) > 0:
        target = match[0]
        break
if target is None:
    target = num.columns[-1]

x = num.drop(columns=target)
y = num[target]
print('target:', target)
print('features:', list(x.columns))

In [ ]:
model = LinearRegression()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
model.fit(x_train, y_train)
pred = model.predict(x_test)
resid = y_test - pred
r2 = r2_score(y_test, pred)
print('test r2:', round(r2, 4))

In [ ]:
cols = list(x.columns)
fig, ax = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))
ax = np.atleast_1d(ax)
for a, c in zip(ax, cols):
    sns.regplot(x=x[c], y=y, ax=a, scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(pred, resid, alpha=0.3)
plt.axhline(0, color='red')
plt.xlabel('predicted')
plt.ylabel('residuals')
plt.show()

plt.figure(figsize=(6, 4))
sns.histplot(resid, kde=True)
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(x.corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
order = np.argsort(pred)
half = len(order) // 2
low_std = resid.values[order[:half]].std()
high_std = resid.values[order[half:]].std()
ratio = high_std / low_std
skew = resid.skew()
corr = x.corr().abs().values.copy()
np.fill_diagonal(corr, 0)
max_corr = corr.max() if corr.size > 0 else 0

print('linearity (test r2):', round(r2, 4))
print('homoscedasticity (std ratio high/low):', round(ratio, 4))
print('normality (residual skew):', round(skew, 4))
print('multicollinearity (max feature correlation):', round(max_corr, 4))

In [ ]:
checks = {
    'linearity': r2 > 0.5,
    'homoscedasticity': 0.67 < ratio < 1.5,
    'normality': abs(skew) < 1,
    'no multicollinearity': max_corr < 0.8,
}
for k, v in checks.items():
    print(k, ':', 'pass' if v else 'fail')

failed = [k for k, v in checks.items() if not v]
if len(failed) == 0:
    print('linear regression is suitable')
else:
    print('linear regression is not fully suitable, violated:', failed)